# Social Doors: download and preprocess one public participant

**Author:** Smith Lab  
**Updated:** 2026-08-28  
**License:** MIT

This notebook retrieves only the data needed for `sub-10317` from OpenNeuro `ds005123` version `1.1.3`, then runs fMRIPrep for the `doors` and `socialdoors` tasks. It contains no private RF1-SRA data.

## What you will learn

- How DataLad can install a dataset without downloading every large file.
- How to restrict fMRIPrep to two tasks and one participant.
- How to identify the optimally combined multi-echo BOLD used by the FEAT lesson.
- How to inspect an fMRIPrep report and a basic image slice.

## Citation and resources

- OpenNeuro: [ds005123 v1.1.3](https://openneuro.org/datasets/ds005123/versions/1.1.3)
- fMRIPrep: Esteban et al. (2019), *Nature Methods*, 16, 111–116.
- DataLad: Halchenko et al. (2021), *Journal of Open Source Software*, 6, 3262.
- Neurodesk: [accessing tools in notebooks](https://neurodesk.org/edu/tutorials/about_neurodesk/accessing_neurodesk_tools.html)

## Table of contents

1. Load software
2. Configure the workspace
3. Install the frozen dataset and retrieve one participant
4. Check the FreeSurfer license
5. Run focused fMRIPrep
6. Inspect outputs and basic QC
7. Record dependencies

## 1. Load software

Neurodesk modules pin the command-line software version. The first load may download a container.

In [ ]:
import module

await module.load('fmriprep/25.2.5')
await module.list()

In [ ]:
%pip install -q nibabel matplotlib watermark

from pathlib import Path
import glob
import json
import os
import subprocess

import matplotlib.pyplot as plt
import nibabel as nib
from IPython.display import IFrame, display

## 2. Configure the workspace

All downloaded/generated material lives outside the Git repository. Change `WORKSPACE` if your Neurodesk installation provides a larger persistent volume.

In [ ]:
SUBJECT = '10317'
SNAPSHOT = '1.1.3'
TASKS = ('doors', 'socialdoors')
WORKSPACE = Path.home() / 'socialdoors_teaching'
BIDS_DIR = WORKSPACE / 'ds005123'
DERIVATIVES_DIR = WORKSPACE / 'derivatives'
SCRATCH_DIR = WORKSPACE / 'scratch' / 'fmriprep'
FILTER_FILE = WORKSPACE / 'bids_filters.json'

for path in (WORKSPACE, DERIVATIVES_DIR, SCRATCH_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f'Workspace: {WORKSPACE}')
print('Warning: fMRIPrep may take several hours on Neurodesk Play.')

## 3. Install the frozen dataset and retrieve one participant

DataLad installs the Git metadata first. `datalad get` then retrieves only the anatomical images, BOLD fieldmaps, and magnitude data for the two task runs (including JSON and events files) needed here. It does not download unrelated RF1 tasks, phase images, or the full dataset. The public OpenNeuro snapshot is sessionless, so its paths are `sub-10317/func/...`, not the production `sub-10317/ses-01/func/...` layout.

In [ ]:
dataset_source = 'https://github.com/OpenNeuroDatasets/ds005123.git'
if not (BIDS_DIR / '.datalad').exists():
    subprocess.run(['datalad', 'install', '-s', dataset_source, str(BIDS_DIR)], check=True)

# OpenNeuro snapshots are Git tags. Detached checkout keeps this exercise frozen.
subprocess.run(['git', '-C', str(BIDS_DIR), 'checkout', SNAPSHOT], check=True)

# ds005123 v1.1.3 is sessionless. Keep these paths tied to the frozen public tree.
patterns = [
    f'sub-{SUBJECT}/anat/*',
    f'sub-{SUBJECT}/fmap/*_acq-bold_*',
]
for task in TASKS:
    patterns.extend([
        f'sub-{SUBJECT}/func/*task-{task}*_part-mag_bold.nii.gz',
        f'sub-{SUBJECT}/func/*task-{task}*_part-mag_bold.json',
        f'sub-{SUBJECT}/func/*task-{task}*_events.tsv',
        f'sub-{SUBJECT}/func/*task-{task}*_events.json',
        f'sub-{SUBJECT}/func/*task-{task}*_part-mag_sbref.nii.gz',
        f'sub-{SUBJECT}/func/*task-{task}*_part-mag_sbref.json',
    ])

targets = sorted({Path(p) for pattern in patterns for p in glob.glob(str(BIDS_DIR / pattern))})
if not targets:
    raise FileNotFoundError(
        f'No matching public files were found for sub-{SUBJECT} at snapshot {SNAPSHOT}. '
        'Expected the sessionless path sub-<ID>/func/.'
    )
relative_targets = [str(path.relative_to(BIDS_DIR)) for path in targets]
subprocess.run(['datalad', 'get', *relative_targets], cwd=BIDS_DIR, check=True)
print(f'Retrieved or verified {len(targets)} files for sub-{SUBJECT}.')

In [ ]:
func_dir = BIDS_DIR / f'sub-{SUBJECT}' / 'func'
for task in TASKS:
    files = sorted(p.name for p in func_dir.glob(f'*task-{task}*') if p.suffix in {'.json', '.tsv', '.gz'})
    print(f'\n{task}: {len(files)} files')
    print('\n'.join(f'  {name}' for name in files))

## 4. Check the FreeSurfer license

Even with `--fs-no-reconall`, fMRIPrep checks for a valid FreeSurfer license. Obtain your own free license from https://surfer.nmr.mgh.harvard.edu/registration.html and save it as `~/.license`. No license text is stored in this notebook.

In [ ]:
FS_LICENSE = Path.home() / '.license'
if not FS_LICENSE.is_file():
    raise FileNotFoundError(
        'FreeSurfer license not found at ~/.license. Obtain your personal license, '
        'save it there, and rerun this cell.'
    )
print(f'Using FreeSurfer license: {FS_LICENSE}')

## 5. Run focused fMRIPrep

The BIDS filter prevents accidental processing of the dataset's other tasks and excludes phase images. `--me-output-echos` retains corrected individual echoes for advanced workflows, while fMRIPrep also produces the optimally combined BOLD used in notebook 02. MNI 2-mm output matches the FEAT teaching workflow. Because this public snapshot is sessionless, the command deliberately omits `--session-label`.

In [ ]:
bids_filter = {
    'bold': {'datatype': 'func', 'task': list(TASKS), 'suffix': 'bold', 'part': [None, 'mag']},
    'sbref': {'datatype': 'func', 'task': list(TASKS), 'suffix': 'sbref', 'part': [None, 'mag']},
    't1w': {'datatype': 'anat', 'suffix': 'T1w'},
    't2w': {'datatype': 'anat', 'suffix': 'T2w'},
    'fmap': {'datatype': 'fmap'},
}
FILTER_FILE.write_text(json.dumps(bids_filter, indent=2) + '\n')
display(bids_filter)

In [ ]:
def combined_bold(task):
    pattern = f'sub-{SUBJECT}/func/*task-{task}*space-MNI152NLin6Asym*desc-preproc_bold.nii.gz'
    return [p for p in DERIVATIVES_DIR.glob(pattern) if 'echo-' not in p.name]

complete = all(len(combined_bold(task)) == 1 for task in TASKS)
if complete:
    print('Complete optimally combined outputs already exist; skipping fMRIPrep.')
else:
    command = [
        'fmriprep', str(BIDS_DIR), str(DERIVATIVES_DIR), 'participant',
        '--participant-label', SUBJECT,
        '--bids-filter-file', str(FILTER_FILE),
        '--output-spaces', 'MNI152NLin6Asym:res-2',
        '--me-output-echos',
        '--fs-no-reconall',
        '--fs-license-file', str(FS_LICENSE),
        '--nprocs', '8', '--omp-nthreads', '2', '--mem-mb', '16000',
        '--stop-on-first-crash',
        '-w', str(SCRATCH_DIR),
    ]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)

## 6. Inspect outputs and basic QC

The HTML report is the primary fMRIPrep QC artifact. Review every section, especially susceptibility correction, registration, and carpet plots. The image below is only a quick sanity check.

In [ ]:
report = DERIVATIVES_DIR / f'sub-{SUBJECT}.html'
if not report.is_file():
    raise FileNotFoundError(f'fMRIPrep report not found: {report}')
display(IFrame(src=str(report), width='100%', height=700))

In [ ]:
outputs = {}
for task in TASKS:
    candidates = combined_bold(task)
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one optimally combined {task} BOLD; found {candidates}')
    outputs[task] = candidates[0]
    print(f'{task}: {candidates[0]}')

img = nib.load(outputs['socialdoors'])
volume = img.dataobj[..., 0]
z = volume.shape[2] // 2
plt.figure(figsize=(7, 6))
plt.imshow(volume[:, :, z].T, cmap='gray', origin='lower')
plt.title(f'sub-{SUBJECT} socialdoors: first preprocessed volume, axial z={z}')
plt.axis('off')
plt.show()
print(f'Shape: {img.shape}; voxel sizes: {img.header.get_zooms()[:3]}')

## 7. Dependencies and handoff

Notebook 02 starts from `BIDS_DIR` and `DERIVATIVES_DIR`, previews the events, creates teaching confounds, and runs the repository's L1 activation model for both tasks.

In [ ]:
%load_ext watermark
%watermark
%watermark --iversions
await module.list()